# Personalized Diet Recommendation System Using K-Means Clustering
## Exploratory Data Analysis, Clustering & Evaluation Notebook
This notebook provides a complete walk-through of data loading, cleaning, exploratory visual analysis, K-Means clustering, silhouette optimization, cluster profiling, and transparent target mapping.

In [ ]:
import sys, os
sys.path.append("..")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_preprocessing import load_dataset, clean_dataset, encode_features, scale_features, ALL_CLUSTERING_FEATURES
from src.clustering import evaluate_kmeans_k, train_kmeans, analyze_clusters, build_cluster_diet_mapping
from src.recommendation import recommend_diet
%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted")

## 1. Load and Inspect Dataset

In [ ]:
df_raw = load_dataset("../data/diet_recommendations_dataset.csv")
print(f"Dataset Shape: {df_raw.shape}")
df_raw.head()

## 2. Preprocessing & Feature Engineering

In [ ]:
df_clean = clean_dataset(df_raw)
df_encoded = encode_features(df_clean)
print("Missing Values after cleaning:")
print(df_clean.isnull().sum())
X = df_encoded[ALL_CLUSTERING_FEATURES]
X_scaled, scaler = scale_features(X)
print(f"Scaled Feature Matrix Shape: {X_scaled.shape}")

## 3. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.histplot(df_clean["Age"], kde=True, ax=axes[0,0], color="royalblue").set_title("Age Distribution")
sns.histplot(df_clean["BMI"], kde=True, ax=axes[0,1], color="teal").set_title("BMI Distribution")
sns.histplot(df_clean["Daily_Caloric_Intake"], kde=True, ax=axes[1,0], color="crimson").set_title("Caloric Intake Distribution")
sns.countplot(x="Physical_Activity_Level", data=df_clean, ax=axes[1,1], palette="dark").set_title("Physical Activity Level")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
num_cols = df_clean.select_dtypes(include=[np.number]).columns
sns.heatmap(df_clean[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Numerical Correlation Heatmap")
plt.show()

## 4. K-Means Clustering & K Evaluation

In [ ]:
eval_df = evaluate_kmeans_k(X_scaled, min_k=2, max_k=8)
display(eval_df)
fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.set_xlabel("Number of Clusters (K)")
ax1.set_ylabel("Inertia (Elbow)", color="tab:blue")
ax1.plot(eval_df["k"], eval_df["inertia"], marker="o", color="tab:blue", linewidth=2)
ax2 = ax1.twinx()
ax2.set_ylabel("Silhouette Score", color="tab:orange")
ax2.plot(eval_df["k"], eval_df["silhouette_score"], marker="s", color="tab:orange", linestyle="--", linewidth=2)
plt.title("K-Means Evaluation Metrics")
plt.show()

## 5. Fit Final Model (K=3) & Cluster Profiling

In [ ]:
kmeans_model, labels, inertia, sil = train_kmeans(X_scaled, n_clusters=3)
df_encoded["Cluster"] = labels
df_clean["Cluster"] = labels
profiles = analyze_clusters(df_encoded, labels, ALL_CLUSTERING_FEATURES)
display(profiles.T)
mapping, summary_df, crosstab, crosstab_pct = build_cluster_diet_mapping(df_clean, labels)
display(summary_df)
display(crosstab_pct)

## 6. Verification with Recommendation System

In [ ]:
sample_user = {
    "Age": 52,
    "Gender": "Male",
    "Weight_kg": 85.0,
    "Height_cm": 172.0,
    "Blood_Pressure_mmHg": 155,
    "Glucose_mg/dL": 145,
    "Cholesterol_mg/dL": 215,
    "Physical_Activity_Level": "Sedentary",
    "Daily_Caloric_Intake": 2800
}
rec = recommend_diet(sample_user, models_dir="../models")
print("Cluster:", rec["cluster_id"])
print("Diet:", rec["recommended_diet"])
print("Insights:", rec["insights"])